# HealthConnect Clinic – Week 4 Initial Analysis


**AnalystLab Africa Experience Lab – Data Analytics**

**Intern:** Adeleke Jubril Adedeji


**Project:** HealthConnect Clinic Experience Lab – Improving Patient Appointment Attendance and Healthcare Support Using Data and AI


**Week 4 Focus:** Problem Understanding – Dataset Overview, Data Quality Assessment, Business Questions, Potential KPIs, and Initial Analysis Approach

## 1. Dataset Overview

This section reviews the structure of the HealthConnect Appointment dataset and confirms it against the HealthConnect Data Dictionary before any analysis is carried out.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\HP Dragon Fly\Downloads\HealthConnect_Appointment_Data.csv")
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [3]:
print(df.shape)
df.info()

(5000, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object

The dataset contains **5,000 appointment records** across **18 columns**, covering patient demographics, appointment details, booking behaviour, reminder information, clinic logistics, and the final appointment outcome. The column names and data types match the HealthConnect Data Dictionary.

## 2. Data Quality Assessment

Before any analysis, the dataset was checked for missing values, duplicates, invalid values, and inconsistencies between related columns.

In [4]:
df.isnull().sum()

appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

Missing values were found in three columns: `reminder_channel` (1,366), `distance_to_clinic_km` (90), and `waiting_time_minutes` (60). All other columns are complete.

The `reminder_channel` gaps were checked against `reminder_sent` to confirm whether they represent a genuine data quality issue or a structural pattern.

In [6]:
print("Duplicate rows:", df.duplicated().sum())
print(df.groupby('reminder_sent')['reminder_channel'].apply(lambda x: x.isnull().sum()))

Duplicate rows: 0
reminder_sent
No     1366
Yes       0
Name: reminder_channel, dtype: int64


There are **no duplicate rows**. All 1,366 missing `reminder_channel` values correspond exactly to appointments where `reminder_sent = No`. This confirms the missing values are **structural, not a data quality issue** — there is simply no channel to record when no reminder was sent.

Next, the `distance_to_clinic_km` and `waiting_time_minutes` columns were checked for realistic ranges, alongside a check of the outcome variable itself.

In [7]:
print(df['appointment_outcome'].value_counts())
print(df['distance_to_clinic_km'].describe())
print(df['waiting_time_minutes'].describe())

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64
count    4910.000000
mean       10.109572
std         6.590030
min         0.500000
25%         5.300000
50%         8.700000
75%        13.500000
max        45.000000
Name: distance_to_clinic_km, dtype: float64
count    4940.000000
mean       24.189676
std        10.863184
min         2.000000
25%        17.000000
50%        24.000000
75%        32.000000
max        68.000000
Name: waiting_time_minutes, dtype: float64


`appointment_outcome` has three categories: **No-Show (2,423)**, **Attended (2,314)**, and **Cancelled (263)**. Both `distance_to_clinic_km` (0.5-45 km) and `waiting_time_minutes` (2-68 minutes) fall within reasonable, realistic ranges with no negative or extreme outlier values.

Two further consistency checks were carried out: whether `previous_no_shows` ever exceeds `previous_appointments` (which should not be possible), and whether all categorical columns contain clean, consistent values.

In [8]:
print("Rows where no_shows > appointments:", (df['previous_no_shows'] > df['previous_appointments']).sum())

for col in ['gender', 'appointment_type', 'appointment_day', 'appointment_time', 'reminder_sent']:
    print(f"\n{col}:")
    print(df[col].unique())

Rows where no_shows > appointments: 0

gender:
['Female' 'Male' 'Prefer not to say']

appointment_type:
['Follow-up' 'Specialist Consultation' 'General Consultation'
 'Diagnostic Test']

appointment_day:
['Tuesday' 'Friday' 'Wednesday' 'Thursday' 'Monday' 'Sunday' 'Saturday']

appointment_time:
['Afternoon' 'Morning' 'Evening']

reminder_sent:
['Yes' 'No']


No rows violate the `previous_no_shows` <= `previous_appointments` rule, and all categorical columns contain clean, expected values with no typos or inconsistent casing.

Finally, the relationship between `booking_date`, `appointment_date`, and `booking_lead_days` was verified for logical consistency.

In [9]:
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])

print("Rows where booking_date is after appointment_date:", (df['booking_date'] > df['appointment_date']).sum())

df['calculated_lead_days'] = (df['appointment_date'] - df['booking_date']).dt.days
print("Rows where booking_lead_days doesn't match calculated gap:", (df['calculated_lead_days'] != df['booking_lead_days']).sum())


Rows where booking_date is after appointment_date: 0
Rows where booking_lead_days doesn't match calculated gap: 0


Both checks returned zero mismatches -- `booking_date` always falls before `appointment_date`, and `booking_lead_days` matches the actual calculated gap in every row.

**Data Quality Summary:** The dataset is clean and reliable. It has no duplicates, no invalid values, and no logical inconsistencies. The only missing values are explainable (`reminder_channel`) or minor and non-critical (`distance_to_clinic_km`, `waiting_time_minutes`).

## 3. Exploring Variables Against Appointment Outcome

Before defining the business questions and KPIs, each variable in the dataset was checked against `appointment_outcome` to identify which factors actually appear to relate to attendance, cancellations, and no-shows.

In [10]:
df.groupby('appointment_outcome')['previous_no_shows'].mean()

appointment_outcome
Attended     0.456785
Cancelled    0.418251
No-Show      0.641354
Name: previous_no_shows, dtype: float64

Patients who eventually no-show have a higher average number of past no-shows (**0.64**) than those who attend (**0.46**) or cancel (**0.42**). Past behaviour appears to carry over.

In [11]:
pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index') * 100

appointment_outcome,Attended,Cancelled,No-Show
reminder_sent,,,
No,42.679356,5.929722,51.390922
Yes,47.633462,5.008255,47.358283


Sending a reminder is associated with a modest reduction in no-shows: **47.4%** no-show rate when a reminder was sent versus **51.4%** when it was not.

In [12]:
print(pd.crosstab(df['appointment_type'], df['appointment_outcome'], normalize='index') * 100)
print()
print(df.groupby('appointment_outcome')['booking_lead_days'].mean())

appointment_outcome       Attended  Cancelled    No-Show
appointment_type                                        
Diagnostic Test          45.868465   4.384486  49.747049
Follow-up                43.349754   5.418719  51.231527
General Consultation     48.274209   5.081496  46.644295
Specialist Consultation  46.555556   6.000000  47.444444

appointment_outcome
Attended     24.521175
Cancelled    29.631179
No-Show      34.526620
Name: booking_lead_days, dtype: float64


**Appointment type** shows mild variation -- Follow-up appointments have the highest no-show rate (51.2%), General Consultation the lowest (46.6%).

**Booking lead time** shows the clearest pattern so far: No-Shows are booked much further in advance (**34.5 days**) than Attended (**24.5 days**) or Cancelled (**29.6 days**) appointments.

In [13]:
print(df.groupby('appointment_outcome')['distance_to_clinic_km'].mean())
print()
print(pd.crosstab(df['age_group'], df['appointment_outcome'], normalize='index') * 100)

appointment_outcome
Attended      9.668703
Cancelled    10.111583
No-Show      10.531481
Name: distance_to_clinic_km, dtype: float64

appointment_outcome   Attended  Cancelled    No-Show
age_group                                           
18-24                45.567376   4.255319  50.177305
25-34                45.083014   4.214559  50.702427
35-44                45.665446   5.982906  48.351648
45-54                46.027743   5.926860  48.045397
55-64                45.000000   4.250000  50.750000
65+                  48.751007   6.124093  45.124899


**Distance to clinic** has almost no effect on outcome (9.7 vs 10.5 km). **Age group** is largely flat across the board, except patients aged **65+** show a notably lower no-show rate (45.1%) than younger groups.

In [14]:
print(pd.crosstab(df['appointment_day'], df['appointment_outcome'], normalize='index') * 100)
print()
print(pd.crosstab(df['gender'], df['appointment_outcome'], normalize='index') * 100)

appointment_outcome   Attended  Cancelled    No-Show
appointment_day                                     
Friday               49.038462   4.395604  46.565934
Monday               43.794579   6.562054  49.643367
Saturday             47.513812   5.662983  46.823204
Sunday               45.047490   4.477612  50.474898
Thursday             45.175439   5.847953  48.976608
Tuesday              49.056604   3.918723  47.024673
Wednesday            44.369064   5.970149  49.660787

appointment_outcome   Attended  Cancelled    No-Show
gender                                              
Female               46.342444   5.225080  48.432476
Male                 45.881864   5.407654  48.710483
Prefer not to say    53.703704   2.777778  43.518519


**Appointment day** and **gender** both show negligible variation in outcome -- neither appears to be a meaningful driver of no-shows.

**Summary -- ranked by apparent influence on appointment outcome:**
1. `booking_lead_days` -- strongest signal
2. `previous_no_shows` -- strong signal
3. `reminder_sent` -- modest effect
4. `age_group` -- flat, except 65+ notably better
5. `appointment_type` -- mild variation
6. `appointment_day`, `gender`, `distance_to_clinic_km` -- negligible effect

## 4. Business Questions

Based on the patterns observed above, the following business questions will guide the next stages of this project:

**Business Question 1:** How does the amount of notice a patient gets when booking -- the gap between the booking date and the actual appointment -- affect whether they show up, cancel, or miss the appointment?

**Business Question 2:** Do patients with a history of missing appointments tend to keep missing them? In other words, does a patient's past no-show record help predict what they'll do next time?

**Business Question 3:** Does sending a reminder actually make a difference in whether a patient shows up, or is its impact smaller than people might assume?

**Business Question 4:** Are older patients more reliable at keeping appointments than younger ones, or does age not really play a role?

**Business Question 5:** Does the type of appointment -- Follow-up, Specialist Consultation, General Consultation, or Diagnostic Test -- affect how likely a patient is to attend, cancel, or miss it?

## 5. Potential KPIs

At this stage, KPIs are only identified and justified based on the patterns found above -- not calculated or visualised. Each KPI is linked to one of the business questions above.

**KPI 1: No-Show Rate by Booking Lead Time Band**
Since no-shows are booked further in advance (34.5 days) than attended appointments (24.5 days), grouping appointments into lead-time bands (e.g. 0-7 days, 8-21 days, 22+ days) and tracking the no-show rate within each band would show where risk climbs, and could support a threshold for extra follow-up on far-out bookings.
*Linked to Business Question 1.*

**KPI 2: No-Show Rate by Prior No-Show History**
Patients who eventually no-show have a higher average number of past no-shows (0.64) than those who attend (0.46). Segmenting patients by prior no-show count and tracking no-show rate per group would show how strongly history predicts future behaviour, useful for flagging high-risk patients for proactive outreach.
*Linked to Business Question 2.*

**KPI 3: No-Show Rate by Reminder Status**
Reminders show a modest effect (51.4% no-show without a reminder vs 47.4% with one). Tracking this KPI over time would help the clinic measure whether reminders are worth the operational cost, and whether a stronger reminder channel might close the gap further.
*Linked to Business Question 3.*

**KPI 4: No-Show Rate by Age Group**
Most age groups sit close together, but 65+ patients show a noticeably lower no-show rate (45.1%) than younger groups (up to 50.8%). Tracking this KPI would help the clinic understand whether younger patients need different engagement strategies.
*Linked to Business Question 4.*

**KPI 5: No-Show Rate by Appointment Type**
Follow-up appointments have the highest no-show rate (51.2%) while General Consultations have the lowest (46.6%). Tracking this KPI would help identify which appointment types need stronger attendance support.
*Linked to Business Question 5.*

## 6. Initial Analysis Approach

The next phase of this project will build directly on the patterns identified in Week 4. The proposed approach is:

1. **Segment appointments by the strongest predictors identified** -- starting with booking lead time and prior no-show history, since these showed the clearest relationship with outcome. This will involve creating lead-time bands and no-show-history groups to calculate the proposed KPIs properly.
2. **Calculate and visualise the 5 proposed KPIs** -- using bar charts and grouped comparisons to make the patterns visible and easy to interpret for stakeholders.
3. **Investigate combined effects** -- for example, whether the reminder effect is stronger or weaker for patients with a high prior no-show count, since factors may interact rather than act independently.
4. **Decide how Cancelled appointments are treated** -- whether they are analysed as their own category, folded into No-Show, or excluded, depending on what best serves the business questions once the analysis goes deeper.
5. **Build toward a dashboard or summary report** in a later week, once the KPI calculations and deeper analysis are complete, to present findings in a way that is useful for clinic decision-making.

This keeps Week 4 strictly at the planning stage while setting up a clear, evidence-grounded path for Week 5.